# Checkpoint 3 – Dimenzijski model podataka

## 1. Odabir poslovnog procesa i granularnosti

Poslovni proces = izvršavanje bankovne transakcije

1 zapis u tablici činjenica = 1 bankovna transakcija

In [8]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

DB_USER = "root"
DB_PASSWORD = "root"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "bank_fraud_db"

connection_string = f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Spojeno na bazu:", result.fetchone()[0])

Spojeno na bazu: bank_fraud_db


In [10]:
tables = pd.read_sql("SHOW TABLES;", engine)
tables

,Tables_in_bank_fraud_db
0,account
1,account_type
2,bank_branch
3,bank_transaction
4,city
5,currency
6,customer
7,device_type
8,merchant
9,merchant_category


In [12]:
query = """
SELECT COUNT(*) AS broj_transakcija
FROM bank_transaction;
"""

pd.read_sql(query, engine)

,broj_transakcija
0,160000


In [14]:
query = """
SELECT *
FROM bank_transaction
LIMIT 5;
"""

pd.read_sql(query, engine)

,transaction_id,transaction_code,account_id,merchant_id,transaction_type_id,transaction_device_id,transaction_location_id,currency_id,transaction_date,transaction_time,transaction_amount,account_balance,is_fraud,transaction_description
0,1,4fa3208f-9e23-42dc-b330-844829d0c12c,1,1,1,1,1,1,2025-01-23,0 days 16:04:07,32415.45,74557.27,0,Bitcoin transaction
1,2,e41c55f9-c016-4ff3-872b-cae72467c75c,2,2,2,2,2,1,2025-01-25,0 days 03:09:52,63062.56,66817.99,0,Mutual fund investment
2,3,af5f667c-d064-4083-bfb7-83396111a3da,3,3,1,3,3,1,2025-01-25,0 days 06:49:53,9711.15,61258.85,0,Seminar registration
3,4,b1355810-d246-4aeb-9932-347f32646172,4,4,1,4,4,1,2025-01-04,0 days 00:53:33,94677.01,36313.61,0,Public transport pass
4,5,c86a000c-d81f-40be-acdf-77fc072fd808,5,5,3,5,5,1,2025-01-16,0 days 04:04:48,67704.28,16948.73,0,Online shopping


## 2. Odabir tablice činjenica i mjera

Središnja tablica dimenzijskog modela: `fact_bank_transaction`

Gl. mjera: `transaction_amount` (predstavlja iznos transakcije i može se agregirati po različitim dimenzijama)

### Mjere u tablici činjenica

| Mjera | Opis |
|---|---|
| transaction_amount | Iznos pojedinačne bankovne transakcije |
| account_balance | Stanje računa u trenutku transakcije |
| is_fraud | Indikator je li transakcija prijevarna |
| transaction_count | Pomoćna mjera s vrijednošću 1 za brojanje transakcija |

In [16]:
dimension_tables = [
    "customer",
    "account",
    "account_type",
    "bank_branch",
    "city",
    "state",
    "merchant",
    "merchant_category",
    "transaction_device",
    "device_type",
    "transaction_type",
    "transaction_location",
    "currency"
]

for table in dimension_tables:
    count_query = f"SELECT COUNT(*) AS broj_redaka FROM {table};"
    result = pd.read_sql(count_query, engine)
    print(table, ":", result.iloc[0, 0])

customer : 160000
account : 160000
account_type : 3
bank_branch : 148
city : 148
state : 34
merchant : 160000
merchant_category : 6
transaction_device : 20
device_type : 4
transaction_type : 5
transaction_location : 148
currency : 1


## 3. Odabir dimenzija

Odabrane dimenzije:

| Dimenzija | Izvorne tablice | Opis |
|---|---|---|
| dim_date | bank_transaction | Vremenska dimenzija izvedena iz datuma transakcije |
| dim_customer | customer | Opisuje korisnika koji je povezan s računom |
| dim_account | account, account_type, bank_branch, city, state | Opisuje račun, tip računa i bankovnu poslovnicu |
| dim_merchant | merchant, merchant_category | Opisuje trgovca i kategoriju trgovca |
| dim_device | transaction_device, device_type | Opisuje uređaj korišten za transakciju |
| dim_transaction_type | transaction_type | Opisuje vrstu transakcije |
| dim_transaction_location | transaction_location | Opisuje lokaciju transakcije |
| dim_currency | currency | Opisuje valutu transakcije |


## 4. Hijerarhije u dimenzijama

### Vremenska

`year → quarter → month → day`

### Račun

`state → city → bank_branch → account_type`

### Trgovac

`merchant_category → merchant`
### Uređaj

`device_type → transaction_device`

In [18]:
query = """
SELECT 
    a.account_id,
    at.account_type_name,
    bb.bank_branch_name,
    c.city_name,
    s.state_name
FROM account a
JOIN account_type at ON a.account_type_id = at.account_type_id
JOIN bank_branch bb ON a.bank_branch_id = bb.bank_branch_id
JOIN city c ON bb.city_id = c.city_id
JOIN state s ON c.state_id = s.state_id
LIMIT 10;
"""

pd.read_sql(query, engine)

,account_id,account_type_name,bank_branch_name,city_name,state_name
0,1,Savings,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
1,289,Savings,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
2,556,Business,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
3,579,Business,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
4,598,Checking,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
5,838,Savings,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
6,995,Checking,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
7,1050,Savings,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
8,1109,Business,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala
9,1124,Business,Thiruvananthapuram Branch,Thiruvananthapuram,Kerala


In [20]:
query = """
SELECT
    m.merchant_id,
    m.merchant_code,
    mc.merchant_category_name
FROM merchant m
JOIN merchant_category mc 
    ON m.merchant_category_id = mc.merchant_category_id
LIMIT 10;
"""

pd.read_sql(query, engine)

,merchant_id,merchant_code,merchant_category_name
0,5,ec344461-1bab-4e85-a1cb-05061e9149bc,Clothing
1,6,1de36b2e-e1a9-4e29-b8f8-8693919f0bb1,Clothing
2,8,e4d6a617-f30f-481c-b2d6-133259d69a3b,Clothing
3,12,9ccd73b9-9361-4ec7-af4f-9bbf27672b77,Clothing
4,15,4a1fff97-40f6-4427-b6fa-a1921619b4df,Clothing
5,16,df14ef22-f7aa-4d72-92ca-3c72c6bb4c7d,Clothing
6,44,b09f4973-11ba-4887-8924-52a7e7f0b2c0,Clothing
7,71,48baaf73-d2e4-470b-8c19-f4e45c63bfe0,Clothing
8,72,046ea0c5-0c94-4357-b675-64eadcd3e7bb,Clothing
9,80,cd533486-f917-478a-96bb-abdd5ce5c143,Clothing


In [22]:
query = """
SELECT 
    mc.merchant_category_name,
    COUNT(*) AS broj_trgovaca
FROM merchant m
JOIN merchant_category mc
    ON m.merchant_category_id = mc.merchant_category_id
GROUP BY mc.merchant_category_name
ORDER BY broj_trgovaca DESC;
"""

pd.read_sql(query, engine)

,merchant_category_name,broj_trgovaca
0,Restaurant,26891
1,Entertainment,26741
2,Electronics,26727
3,Clothing,26651
4,Health,26507
5,Groceries,26483


In [24]:
query = """
SELECT
    td.transaction_device_id,
    td.transaction_device_name,
    dt.device_type_name
FROM transaction_device td
JOIN device_type dt
    ON td.device_type_id = dt.device_type_id
LIMIT 10;
"""

pd.read_sql(query, engine)

,transaction_device_id,transaction_device_name,device_type_name
0,5,Debit/Credit Card,ATM
1,15,Smart Card,ATM
2,18,Virtual Card,ATM
3,20,Banking Chatbot,ATM
4,2,ATM,Desktop
5,4,Payment Gateway Device,Desktop
6,6,Bank Branch,Desktop
7,9,Biometric Scanner,Desktop
8,10,POS Mobile Device,Desktop
9,11,Self-service Banking Machine,Desktop


In [26]:
query = """
SELECT 
    dt.device_type_name,
    COUNT(*) AS broj_uredaja
FROM transaction_device td
JOIN device_type dt
    ON td.device_type_id = dt.device_type_id
GROUP BY dt.device_type_name
ORDER BY broj_uredaja DESC;
"""

pd.read_sql(query, engine)

,device_type_name,broj_uredaja
0,Desktop,8
1,ATM,4
2,Mobile,4
3,POS,4


In [28]:
query = """
SELECT
    customer_id,
    customer_code,
    customer_name,
    gender,
    age,
    customer_contact,
    customer_email
FROM customer
LIMIT 10;
"""

pd.read_sql(query, engine)

,customer_id,customer_code,customer_name,gender,age,customer_contact,customer_email
0,1,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,+9198579XXXXXX,oshaXXXXX@XXXXX.com
1,2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,+9197745XXXXXX,ekaniXXX@XXXXXX.com
2,3,6c870d65-76b0-431d-bdf3-9292998e8211,Ishanvi Dar,Male,54,+9198318XXXXXX,ishanviXXX@XXXXX.com
3,4,5323737c-bbd2-423f-9c9b-e0433c8f75dc,Arya Shroff,Female,61,+9194785XXXXXX,aryaXXX@XXXXX.com
4,5,c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310,Jackson Shere,Male,32,+9193423XXXXXX,jacksonXXX@XXXXXXX.com
5,6,e9a82764-1253-4a46-ad34-80e3416fc801,Bhanumati Ravel,Male,52,+9194374XXXXXX,bhanumatiXXXXX@XXXXX.com
6,7,708224d5-192a-4d86-b411-8ec6d1bb274b,Meera Ganesh,Female,32,+9194511XXXXXX,meeraXXXXX@XXXXXXX.com
7,8,87b53799-9959-4534-9aad-f7dfc1821d4b,Charvi Biswas,Male,18,+9193360XXXXXX,charviXXXX@XXXXXXX.com
8,9,d4d0f62e-00b0-4c0e-8188-0b5a3f3e6c3e,Yuvraj Dasgupta,Male,59,+9193342XXXXXX,yuvrajXXXXX@XXXXX.com
9,10,b2563484-0dae-4555-8859-5b55f91136d4,Amara Sidhu,Male,66,+9192865XXXXXX,amaraXXXX@XXXXXXX.com


## 5. Napredne mogućnosti dimenzijskog modela

In [30]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

DB_USER = "root"
DB_PASSWORD = "root"
DB_HOST = "localhost"
DB_PORT = 3306

server_connection_string = f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}"
server_engine = create_engine(server_connection_string)

with server_engine.connect() as connection:
    connection.execute(text("CREATE DATABASE IF NOT EXISTS bank_fraud_dw;"))
    connection.commit()

print("Baza bank_fraud_dw je kreirana ili već postoji.")

Baza bank_fraud_dw je kreirana ili već postoji.


In [32]:
DW_NAME = "bank_fraud_dw"

dw_connection_string = f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DW_NAME}"

dw_engine = create_engine(dw_connection_string)

with dw_engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Spojeno na DW bazu:", result.fetchone()[0])

Spojeno na DW bazu: bank_fraud_dw


## 6. Kreiranje star sheme u bazi podataka

In [34]:
drop_tables = [
    "fact_bank_transaction",
    "dim_currency",
    "dim_transaction_location",
    "dim_transaction_type",
    "dim_device",
    "dim_merchant",
    "dim_account",
    "dim_customer",
    "dim_date"
]

create_tables = [

"""
CREATE TABLE dim_date (
    date_tk INT AUTO_INCREMENT PRIMARY KEY, 
    full_date DATE NOT NULL,
    day_of_month INT,
    month_number INT,
    month_name VARCHAR(20),
    quarter_number INT,
    year_number INT,
    UNIQUE KEY uq_dim_date_full_date (full_date)
);
""",

"""
CREATE TABLE dim_customer (
    customer_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    customer_code VARCHAR(100),
    customer_name VARCHAR(100),
    gender VARCHAR(10),
    age INT,
    age_group VARCHAR(20),
    customer_contact VARCHAR(50),
    customer_email VARCHAR(100),
    version INT DEFAULT 1,
    date_from DATE,
    date_to DATE,
    is_current TINYINT(1) DEFAULT 1
);
""",

"""
CREATE TABLE dim_account (
    account_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    account_id INT NOT NULL,
    account_type_name VARCHAR(50),
    bank_branch_name VARCHAR(100),
    city_name VARCHAR(100),
    state_name VARCHAR(100),
    version INT DEFAULT 1,
    date_from DATE,
    date_to DATE,
    is_current TINYINT(1) DEFAULT 1
);
""",

"""
CREATE TABLE dim_merchant (
    merchant_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    merchant_id INT NOT NULL,
    merchant_code VARCHAR(100),
    merchant_category_name VARCHAR(100),
    version INT DEFAULT 1,
    date_from DATE,
    date_to DATE,
    is_current TINYINT(1) DEFAULT 1
);
""",

"""
CREATE TABLE dim_device (
    device_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    transaction_device_id INT NOT NULL,
    transaction_device_name VARCHAR(100),
    device_type_name VARCHAR(100)
);
""",

"""
CREATE TABLE dim_transaction_type (
    transaction_type_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    transaction_type_id INT NOT NULL,
    transaction_type_name VARCHAR(50)
);
""",

"""
CREATE TABLE dim_transaction_location (
    transaction_location_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    transaction_location_id INT NOT NULL,
    transaction_location_name VARCHAR(150)
);
""",

"""
CREATE TABLE dim_currency (
    currency_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    currency_id INT NOT NULL,
    currency_code VARCHAR(10)
);
""",

"""
CREATE TABLE fact_bank_transaction (
    fact_transaction_tk BIGINT AUTO_INCREMENT PRIMARY KEY,
    transaction_id INT NOT NULL,
    transaction_code VARCHAR(100),

    date_tk INT,
    customer_tk BIGINT,
    account_tk BIGINT,
    merchant_tk BIGINT,
    device_tk BIGINT,
    transaction_type_tk BIGINT,
    transaction_location_tk BIGINT,
    currency_tk BIGINT,

    transaction_time TIME,
    transaction_amount DECIMAL(12,2),
    account_balance DECIMAL(12,2),
    is_fraud TINYINT(1),
    transaction_count INT DEFAULT 1,
    transaction_description VARCHAR(255),

    FOREIGN KEY (date_tk) REFERENCES dim_date(date_tk),
    FOREIGN KEY (customer_tk) REFERENCES dim_customer(customer_tk),
    FOREIGN KEY (account_tk) REFERENCES dim_account(account_tk),
    FOREIGN KEY (merchant_tk) REFERENCES dim_merchant(merchant_tk),
    FOREIGN KEY (device_tk) REFERENCES dim_device(device_tk),
    FOREIGN KEY (transaction_type_tk) REFERENCES dim_transaction_type(transaction_type_tk),
    FOREIGN KEY (transaction_location_tk) REFERENCES dim_transaction_location(transaction_location_tk),
    FOREIGN KEY (currency_tk) REFERENCES dim_currency(currency_tk)
);
"""

]

with dw_engine.begin() as connection:
    for table in drop_tables:
        connection.execute(text(f"DROP TABLE IF EXISTS {table};"))
    
    for statement in create_tables:
        connection.execute(text(statement))

print("Star shema je uspješno kreirana u bazi bank_fraud_dw.")

Star shema je uspješno kreirana u bazi bank_fraud_dw.


In [36]:
dw_tables = pd.read_sql("SHOW TABLES;", dw_engine)
dw_tables

,Tables_in_bank_fraud_dw
0,dim_account
1,dim_currency
2,dim_customer
3,dim_date
4,dim_device
5,dim_merchant
6,dim_transaction_location
7,dim_transaction_type
8,fact_bank_transaction


## 7. Prikaz star sheme

<img src="star_schema.png" alt="Star shema" width="700">